# Notebook 6: Evaluation Metrics

Compute and visualize evaluation metrics for attacks and defenses.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from helpers import (load_scenario, get_all_detections, get_ground_truth,
                     CameraAttackerDF, PointCloudAttackerDF, FusionAttackerDF,
                     DefensePipelineDF, CertifiedDefenseDF)
from attacks.camera_attacks import AttackType
from attacks.radar_lidar_attacks import PointCloudAttackType
from attacks.fusion_attacks import FusionAttackType
from defenses.defense_mechanisms import DefenseType

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)

## 6.1 Load Data & Run Attacks

In [ ]:
SCENARIO = 'scenario2'
loader = load_scenario(SCENARIO)
detections = get_all_detections(loader)
ground_truth = get_ground_truth(loader)

# Run camera attack
cam = CameraAttackerDF(epsilon=0.05)
ir_attacked = cam.attack_detections(detections[3].copy(), AttackType.FGSM, sensor_id=3)

# Run point cloud attack
pc = PointCloudAttackerDF(epsilon=5.0)
lidar_attacked = pc.attack_detections(detections[1].copy(), PointCloudAttackType.GHOST_INJECTION, sensor_id=1)

# Run fusion attack
fus = FusionAttackerDF()
fusion_attacked = fus.attack_scenario(detections.copy(), FusionAttackType.SENSOR_DOS, ground_truth)

print('All attacks executed')

## 6.2 Compute Metrics

In [ ]:
from helpers import compute_sensor_metrics

# Benign metrics
benign_ir = compute_sensor_metrics(detections[3], ground_truth, 3)
benign_lidar = compute_sensor_metrics(detections[1], ground_truth, 1)

# Attacked metrics
att_ir = compute_sensor_metrics(ir_attacked, ground_truth, 3)
att_lidar = compute_sensor_metrics(lidar_attacked, ground_truth, 1)

print('=== IR Camera ===')
print('Benign:', benign_ir)
print('Attacked:', att_ir)
print()
print('=== Lidar ===')
print('Benign:', benign_lidar)
print('Attacked:', att_lidar)

## 6.3 Defense Metrics

In [ ]:
# Apply certified defense
cert = CertifiedDefenseDF(certified_radius=0.05)
cert_ir = cert.defend({'3': ir_attacked.copy()}, ground_truth)
cert_metrics = compute_sensor_metrics(cert_ir['3'], ground_truth, 3)

# Apply defense pipeline
pipeline = DefensePipelineDF()
def_ir = pipeline.defend_detections({'3': ir_attacked.copy()}, DefenseType.INPUT_SANITIZATION, ground_truth)
def_metrics = compute_sensor_metrics(def_ir['3'], ground_truth, 3)

print('Certified defense:', cert_metrics)
print('Input sanitization:', def_metrics)

## 6.4 Metric Comparison Bar Chart

In [ ]:
metrics = ['detection_probability', 'false_alarm_rate', 'rmse']
scenarios = ['Benign', 'Attacked', 'Certified', 'Sanitized']

ir_values = {
    'Benign': [benign_ir[m] for m in metrics],
    'Attacked': [att_ir[m] for m in metrics],
    'Certified': [cert_metrics[m] for m in metrics],
    'Sanitized': [def_metrics[m] for m in metrics]
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, metric in enumerate(metrics):
    ax = axes[idx]
    values = [ir_values[s][idx] for s in scenarios]
    bars = ax.bar(scenarios, values, color=['green', 'red', 'blue', 'orange'])
    ax.set_title(metric.replace('_', ' ').title())
    ax.set_ylabel('Value')
    ax.grid(True, axis='y')
    
    # Clip RMSE for visualization
    if metric == 'rmse':
        ax.set_ylim(0, min(max(values) * 1.2, 500))

plt.suptitle('IR Camera: Metrics Comparison', fontsize=14)
plt.tight_layout()
plt.show()

## 6.5 Attack Success Rate

In [ ]:
attack_types = [AttackType.FGSM, AttackType.PGD, AttackType.BIM, AttackType.CW]
success_rates = []

for atype in attack_types:
    attacked = cam.attack_detections(detections[3].copy(), atype, sensor_id=3)
    metrics = compute_sensor_metrics(attacked, ground_truth, 3)
    # Success = increase in RMSE or decrease in detection probability
    success = (metrics['rmse'] - benign_ir['rmse']) / max(benign_ir['rmse'], 1e-6)
    success_rates.append(min(1.0, max(0, success)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([a.name for a in attack_types], success_rates, color='coral')
ax.set_ylabel('Attack Success Rate')
ax.set_title('Camera Attack Success Rates')
ax.set_ylim(0, 1)
ax.grid(True, axis='y')
plt.tight_layout()
plt.show()